# 006 · array and memoryview

> **Standard Library › Data Structures** · Python 3.14

A Python `list` stores **references** to full Python objects, which is flexible but memory-hungry for large amounts of numbers. The `array` module gives you compact, typed arrays of machine values (like `int[]` or `double[]`), and `memoryview` lets you look at and slice binary buffers **without copying** — Python's take on `Span<T>` / `Memory<T>`.

**In this notebook you will learn:**

- `array.array` and its type codes, and how it compares to `int[]` / `double[]`
- converting arrays to and from raw bytes
- a quick recap of `bytes` and `bytearray`
- `memoryview` for zero-copy slicing, writing through views, and reinterpreting (`cast`) buffers
- when a plain `list` is still the right choice

## .NET mental model

| .NET | Python |
|---|---|
| `int[]`, `double[]`, `byte[]` (value-type arrays) | `array.array("i", ...)`, `array.array("d", ...)`, `bytearray` |
| `List<object>` (boxed values) | `list` — every element is a separate object |
| `Span<T>` / `ReadOnlySpan<T>` | `memoryview` (writable / read-only) |
| `Memory<T>` (storable, async-friendly) | `memoryview` (a normal object, can be stored anywhere) |
| `MemoryMarshal.Cast<byte, int>(span)` | `memoryview.cast("i")` |
| `MemoryMarshal.AsBytes(span)` | `memoryview.cast("B")` / `arr.tobytes()` (copy) |

There is no `Span<T>` stack-only restriction in Python: a `memoryview` is an ordinary heap object.

## `array.array`: typed, compact arrays

`array.array(typecode, initializer)` creates a mutable sequence that stores raw C values of a single type. It supports almost the whole `list` API — indexing, slicing, `append`, `extend`, `insert`, `pop`, iteration — but every element must fit the type.

```csharp
// C#
int[] ids = { 1, 2, 3 };
double[] readings = { 20.5, 21.0 };
```

In [ ]:
from array import array

ids = array("i", [1, 2, 3])          # signed int (usually 4 bytes)
readings = array("d", [20.5, 21.0])  # double (8 bytes)

ids.append(4)
ids.extend([5, 6])
print(ids, ids[0], ids[-2:])
print(readings, sum(readings) / len(readings))
print(ids.typecode, ids.itemsize)    # element type and size in bytes

### Type codes

| Code | C type | Typical size | .NET |
|---|---|---|---|
| `"b"` / `"B"` | signed / unsigned char | 1 | `sbyte` / `byte` |
| `"h"` / `"H"` | short / unsigned short | 2 | `short` / `ushort` |
| `"i"` / `"I"` | int / unsigned int | 4 | `int` / `uint` |
| `"l"` / `"L"` | long / unsigned long | 4 or 8 (platform!) | — |
| `"q"` / `"Q"` | long long / unsigned long long | 8 | `long` / `ulong` |
| `"f"` / `"d"` | float / double | 4 / 8 | `float` / `double` |
| `"w"` | Unicode code point (3.13+) | 4 | `Rune` |

Sizes are platform-dependent C sizes — check `itemsize` if it matters, and prefer `"q"` over `"l"` for 64-bit integers (`"l"` is 4 bytes on Windows).

In [ ]:
import array as array_module

print(array_module.typecodes)
for code in "bhiqfd":
    print(code, array(code).itemsize)

⚠️ **Gotcha:** unlike a list, an array **enforces its element type** — at runtime, not just for type checkers. Wrong types raise `TypeError`, out-of-range values raise `OverflowError` (no silent wrap-around like an `unchecked` C# cast).

In [ ]:
try:
    ids.append(2.5)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

try:
    array("b", [200])               # signed byte: -128..127
except OverflowError as e:
    print(f"{type(e).__name__}: {e}")

## Why bother? Memory

A list of a million floats holds a million pointers **plus** a million separate float objects (24 bytes each). An `array("d")` stores just the 8-byte values back to back — like `double[]` vs `List<object>`.

In [ ]:
import sys

n = 100_000
as_list = [float(i) for i in range(n)]
as_array = array("d", as_list)

list_bytes = sys.getsizeof(as_list) + sum(sys.getsizeof(x) for x in as_list)
array_bytes = sys.getsizeof(as_array)
print(f"list:  {list_bytes:>9,} bytes")
print(f"array: {array_bytes:>9,} bytes")
print(f"ratio: {list_bytes / array_bytes:.1f}x")

## Arrays and raw bytes

Arrays convert to and from raw bytes in the machine's native byte order — a quick way to read/write binary numeric data (like `MemoryMarshal.AsBytes` or `Buffer.BlockCopy`). Use `byteswap()` to flip endianness; for structured binary formats the `struct` module is usually a better fit.

In [ ]:
values = array("h", [1, 256, -1])     # 2-byte shorts
raw = values.tobytes()
print(raw, len(raw), sys.byteorder)

restored = array("h")
restored.frombytes(raw)
print(restored)

values.byteswap()                     # convert to the other endianness, in place
print(values.tobytes())

## Recap: `bytes` and `bytearray`

`bytes` (immutable) and `bytearray` (mutable) are effectively typed arrays with type code `"B"` — sequences of ints 0–255. They were covered in detail in *(see Standard Library › Text Processing › 004 · Unicode, Bytes and Encodings)*.

In [ ]:
packet = bytearray(b"\x01\x02\x03\x04")
packet[0] = 0xFF
packet += b"\x05"
print(packet, list(packet), packet.hex(" "))
print(array("B", packet).tolist() == list(packet))

## `memoryview`: zero-copy views

Slicing `bytes`, `bytearray` or an `array` **copies** the data — just like `array[1..3]` on a C# array allocates a new array. A `memoryview` wraps any object that supports the *buffer protocol* and gives you slices that share the original memory, exactly like `Span<T>.Slice`.

```csharp
// C#
byte[] buffer = "hello world"u8.ToArray();
Span<byte> word = buffer.AsSpan(6);   // no copy
word[0] = (byte)'W';                  // writes into buffer
```

In [ ]:
buffer = bytearray(b"hello world")

copy = buffer[6:]                 # a new bytearray (copied)
view = memoryview(buffer)[6:]     # a window onto the same memory

view[0] = ord("W")                # write through the view
print(buffer)                     # changed!
print(copy)                       # the copy is unaffected
print(view.tobytes(), view.nbytes)

Views over **immutable** objects (`bytes`) are read-only — like `ReadOnlySpan<T>`:

In [ ]:
ro = memoryview(b"abc")
print(ro.readonly)
try:
    ro[0] = 65
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

### Why zero-copy matters

Parsing a large binary message by repeatedly slicing `bytes` creates a new copy at every step. Slicing a `memoryview` is O(1) regardless of size. Many I/O APIs (`file.readinto`, `socket.recv_into`) also accept a writable `memoryview`, so you can fill part of a pre-allocated buffer without temporary objects.

In [ ]:
import timeit

data = bytes(10_000_000)          # 10 MB of zeros
mv = memoryview(data)

t_bytes = timeit.timeit(lambda: data[1:], number=20)
t_view = timeit.timeit(lambda: mv[1:], number=20)
print(f"bytes slice:      {t_bytes:.4f}s")
print(f"memoryview slice: {t_view:.6f}s")
print("view is faster:", t_view < t_bytes)    # exact numbers vary by machine

### Views over arrays, and `cast`

A `memoryview` knows the element format of the underlying buffer. `cast` reinterprets the same bytes as another type — the equivalent of `MemoryMarshal.Cast<TFrom, TTo>`.

In [ ]:
nums = array("i", [1, 2, 3, 4])
mv = memoryview(nums)
print(mv.format, mv.itemsize, len(mv), mv.nbytes)
print(mv[1:3].tolist())

as_bytes = mv.cast("B")           # same memory, viewed as unsigned bytes
print(len(as_bytes), as_bytes[:4].tolist())   # the int 1, little-endian

mv[0] = 100                       # writes through to the array
print(nums)

⚠️ **Gotcha:** while a `memoryview` exists, the underlying `bytearray`/`array` **can't be resized** (its memory might move — .NET avoids this problem because a `Span` can't outlive the stack frame). Release the view with `.release()` or a `with` block.

In [ ]:
buf = bytearray(b"abc")
mv = memoryview(buf)
try:
    buf.append(ord("d"))
except BufferError as e:
    print(f"{type(e).__name__}: {e}")

mv.release()
buf.append(ord("d"))              # fine now
with memoryview(buf) as v:        # released automatically at the end of the block
    print(v.tobytes())

## When to use what

| Situation | Use |
|---|---|
| General-purpose collection, mixed types, small/medium data | `list` |
| Large homogeneous numeric data, memory matters | `array.array` |
| Raw binary data | `bytes` / `bytearray` |
| Slicing big buffers without copying, I/O into pre-allocated buffers | `memoryview` |
| Heavy numeric computation (vector math, matrices) | the third-party **NumPy** — `array` has no vectorised maths |

**Tip:** `array` saves memory but **not** CPU time: every element you read is converted back into a Python object, so loops over an `array` are no faster than over a `list`. Reach for it when memory or binary layout matters.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Typed int array | `int[]` | `array("i", [...])` |
| Typed double array | `double[]` | `array("d", [...])` |
| 64-bit ints | `long[]` | `array("q", [...])` |
| Element size | `sizeof(int)` | `arr.itemsize` |
| To raw bytes | `MemoryMarshal.AsBytes(span)` | `arr.tobytes()` |
| From raw bytes | `MemoryMarshal.Cast<byte, T>` | `arr.frombytes(b)` |
| Mutable byte buffer | `byte[]` | `bytearray` |
| Zero-copy slice | `span.Slice(i, n)` | `memoryview(buf)[i:i+n]` |
| Read-only view | `ReadOnlySpan<T>` | `memoryview(bytes_obj)` |
| Reinterpret type | `MemoryMarshal.Cast` | `mv.cast("i")` |

## Exercises

1. Store the squares of 0–9 in an `array("q")`, then convert it to bytes and back into a new array, and check the two are equal.
2. Create a `bytearray` containing `b"HEADERpayload"` and use a `memoryview` to uppercase the payload part **in place** (without slicing a copy of the whole buffer).
3. Use `memoryview.cast` to view `array("i", [1, 2, 3])` as unsigned bytes and print how many bytes it occupies.
4. Try to store `-1` in an `array("B")`. What happens and why?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from array import array

# 1
squares = array("q", (i * i for i in range(10)))
copy = array("q")
copy.frombytes(squares.tobytes())
print(copy == squares, copy)

# 2 — write through a view of the payload part
buf = bytearray(b"HEADERpayload")
payload = memoryview(buf)[6:]
payload[:] = bytes(payload).upper()     # slice assignment of the same length
print(buf)

# 3
mv = memoryview(array("i", [1, 2, 3])).cast("B")
print(len(mv), mv.nbytes)

# 4 — "B" is unsigned (0..255), so -1 raises OverflowError
try:
    array("B", [-1])
except OverflowError as e:
    print(f"{type(e).__name__}: {e}")
```

</details>